# Platoon & Situational Analysis

This notebook examines:
- Platoon splits (vs LHB vs RHB)
- Performance through the batting order
- High leverage performance
- Inning-by-inning breakdown

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'
PITCHERS = {
    'Ohtani': 'shohei_ohtani.csv',
    'Sanchez': 'cristopher_sanchez.csv',
    'Misiorowski': 'jacob_misiorowski.csv'
}

dfs = {}
for name, fname in PITCHERS.items():
    df = pd.read_csv(f'{DATA_DIR}/{fname}')
    df['player_name_clean'] = df['player_name'].str.replace('\n', ', ', regex=False)
    dfs[name] = df

In [ ]:
# Platoon splits
print('PLATOON SPLITS')
print('=' * 120)
platoon_rows = []
for name, df in dfs.items():
    for stand in ['L', 'R']:
        sub = df[df['stand'] == stand]
        if len(sub) < 10:
            continue
        pa_events = sub[sub['events'].notna() & (sub['events'] != '')]
        total_pa = len(pa_events)
        strikeouts = len(pa_events[pa_events['events'] == 'strikeout'])
        walks = len(pa_events[pa_events['events'] == 'walk'])
        singles = len(pa_events[pa_events['events'] == 'single'])
        doubles = len(pa_events[pa_events['events'] == 'double'])
        triples = len(pa_events[pa_events['events'] == 'triple'])
        homers = len(pa_events[pa_events['events'] == 'home_run'])
        hits = singles + doubles + triples + homers
        avg = hits / total_pa if total_pa > 0 else 0
        obp = (hits + walks) / total_pa if total_pa > 0 else 0
        slg = (singles + 2*doubles + 3*triples + 4*homers) / total_pa if total_pa > 0 else 0
        platoon_rows.append({
            'Pitcher': name, 'vs': stand, 'PA': total_pa,
            'K%': f'{strikeouts/total_pa*100:.1f}%',
            'BB%': f'{walks/total_pa*100:.1f}%',
            'AVG': f'{avg:.3f}', 'OPS': f'{obp+slg:.3f}',
            'HR': homers
        })
platoon_df = pd.DataFrame(platoon_rows)
print(platoon_df.to_string(index=False))

In [ ]:
# Performance by time through order
print('PERFORMANCE BY TIME THROUGH ORDER')
print('=' * 120)
tto_rows = []
for name, df in dfs.items():
    tto_col = 'n_thruorder_pitcher'
    if tto_col not in df.columns:
        continue
    for tto in sorted(df[tto_col].dropna().unique()):
        sub = df[df[tto_col] == tto]
        pa_events = sub[sub['events'].notna() & (sub['events'] != '')]
        total_pa = len(pa_events)
        if total_pa == 0:
            continue
        strikeouts = len(pa_events[pa_events['events'] == 'strikeout'])
        walks = len(pa_events[pa_events['events'] == 'walk'])
        singles = len(pa_events[pa_events['events'] == 'single'])
        doubles = len(pa_events[pa_events['events'] == 'double'])
        homers = len(pa_events[pa_events['events'] == 'home_run'])
        hits = singles + doubles + len(homers)
        avg = hits / total_pa if total_pa > 0 else 0
        tto_rows.append({
            'Pitcher': name, 'TTO': int(tto), 'PA': total_pa,
            'K%': f'{strikeouts/total_pa*100:.1f}%',
            'BB%': f'{walks/total_pa*100:.1f}%',
            'AVG': f'{avg:.3f}'
        })
tto_df = pd.DataFrame(tto_rows)
print(tto_df.to_string(index=False))

In [ ]:
# Platoon OPS visualization
fig, ax = plt.subplots(figsize=(10, 6))
names = list(dfs.keys())
colors = {'Ohtani': '#1f77b4', 'Sanchez': '#ff7f0e', 'Misiorowski': '#2ca02c'}
x = np.arange(len(names))
w = 0.35
for i, name in enumerate(names):
    lhb_df = platoon_df[(platoon_df['Pitcher'] == name) & (platoon_df['vs'] == 'L')]
    rhb_df = platoon_df[(platoon_df['Pitcher'] == name) & (platoon_df['vs'] == 'R')]
    lhb_val = float(lhb_df['OPS'].iloc[0]) if len(lhb_df) > 0 else 0
    rhb_val = float(rhb_df['OPS'].iloc[0]) if len(rhb_df) > 0 else 0
    ax.bar(i - w/2, lhb_val, w, label='vs LHB' if i == 0 else '', color='#d62728', alpha=0.8)
    ax.bar(i + w/2, rhb_val, w, label='vs RHB' if i == 0 else '', color='#1f77b4', alpha=0.8)
    ax.text(i - w/2, lhb_val + 0.02, f'{lhb_val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + w/2, rhb_val + 0.02, f'{rhb_val:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('OPS Allowed')
ax.set_title('Platoon Splits: OPS Allowed vs LHB/RHB', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('figures/platoon_splits.png', dpi=150, bbox_inches='tight')
plt.show()